In [2]:
!pip -q install -U "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 90.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.4 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 r

In [3]:
#Config + inlined helpers
import os, json, re
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DATA = "/kaggle/input/datasets/ianrusseladem/track-c-data"
OUT  = "/kaggle/working"
MAX_LEN, BATCH, GRAD_ACCUM, EPOCHS = 1408, 1, 16, 3
LABELS = ["Done", "Won't Do"]

def read_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]
def normalize(s): return " ".join(s.lower().split())
def c_predict_label(output, labels):
    tail = output.rsplit("</think>", 1)[-1] if "</think>" in output else output
    by = {normalize(l): l for l in labels}
    lines = [x for x in tail.splitlines() if x.strip()]
    last = normalize(lines[-1]) if lines else ""
    if last in by: return by[last]
    low = normalize(tail); hits = [l for l in labels if normalize(l) in low]
    return max(hits, key=len) if hits else None
def macro_f1(gold, pred, labels):
    tot = 0.0
    for l in labels:
        tp = sum(1 for g,p in zip(gold,pred) if g==l and p==l)
        fp = sum(1 for g,p in zip(gold,pred) if g!=l and p==l)
        fn = sum(1 for g,p in zip(gold,pred) if g==l and p!=l)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        tot += 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return tot/len(labels)
print("config ready; DATA contents:", os.listdir(DATA) if os.path.isdir(DATA) else "ADD THE DATASET")

config ready; DATA contents: ['train_mix.jsonl', 'sentinel.jsonl', 'seed.jsonl', 'gold.jsonl', 'reasoning_probes.jsonl', 'tool_probes.jsonl']


In [4]:
#Train (control = seed, real = seed-synth)
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_one(data_file, name):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto"); model.config.use_cache = False
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules="all-linear")
    ds = load_dataset("json", data_files=f"{DATA}/{data_file}", split="train")
    out = f"{OUT}/lora-{name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=2e-4, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy="epoch", bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=MAX_LEN, packing=False, assistant_only_loss=True, seed=0, report_to="none")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok)
    r = tr.train(); tr.save_model(out); tok.save_pretrained(out)
    print(f"{name}: final_train_loss={r.training_loss:.4f} -> {out}"); return out

train_one("seed.jsonl", "seed")
train_one("train_mix.jsonl", "seed-synth")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/296 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.165792
20,2.220651
30,2.005195


seed-synth: final_train_loss=2.4639 -> /kaggle/working/lora-seed-synth


'/kaggle/working/lora-seed-synth'

In [5]:
#Evaluate base vs seed vs seed-synth (both axes)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model(adapter=None):
    t = AutoTokenizer.from_pretrained(BASE_MODEL); t.padding_side = "left"
    if t.pad_token is None: t.pad_token = t.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), t

@torch.no_grad()
def gen(m, t, prompts, max_new, batch=16):
    outs = []
    for i in range(0, len(prompts), batch):
        ch = prompts[i:i+batch]
        txt = [t.apply_chat_template(x, add_generation_prompt=True, tokenize=False) for x in ch]
        enc = t(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=t.pad_token_id)
        for j in range(len(ch)):
            outs.append(t.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

def probe_score(m, t, fname, mode, max_new):
    rows = read_jsonl(f"{DATA}/{fname}")
    raw = gen(m, t, [[{"role":"user","content":r["question"]}] for r in rows], max_new)
    ok = 0
    for r, o in zip(rows, raw):
        ans = [a.lower() for a in r["answers"]]; low = o.lower()
        ok += (all(a in low for a in ans) if mode=="all" else any(a in low for a in ans))
    return ok/len(rows)

def evaluate(name, adapter=None):
    gold = read_jsonl(f"{DATA}/gold.jsonl")
    m, t = load_model(adapter)
    raw = gen(m, t, [r["messages"] for r in gold], 384)
    g = [r["label"] for r in gold]; p = [c_predict_label(o, LABELS) for o in raw]
    acc = sum(1 for a,b in zip(g,p) if a==b)/len(g); f1 = macro_f1(g, p, LABELS)
    sent = probe_score(m, t, "sentinel.jsonl", "any", 32)
    reason = probe_score(m, t, "reasoning_probes.jsonl", "any", 256)
    tools = probe_score(m, t, "tool_probes.jsonl", "all", 128)
    res = {"name":name, "acc":round(acc,3), "macro_f1":round(f1,3),
           "sentinel":round(sent,3), "reasoning":round(reason,3), "tools":round(tools,3)}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2); print(res)
    del m; torch.cuda.empty_cache(); return res

R = [evaluate("base"), evaluate("seed", f"{OUT}/lora-seed"), evaluate("seed-synth", f"{OUT}/lora-seed-synth")]
print("\nname           acc    macroF1  sentinel  reasoning  tools")
for r in R:
    print(f"{r['name']:<14} {r['acc']:.3f}  {r['macro_f1']:.3f}    {r['sentinel']:.3f}     {r['reasoning']:.3f}      {r['tools']:.3f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'name': 'base', 'acc': 0.515, 'macro_f1': 0.395, 'sentinel': 0.917, 'reasoning': 0.8, 'tools': 0.75}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'name': 'seed', 'acc': 0.515, 'macro_f1': 0.367, 'sentinel': 0.833, 'reasoning': 0.9, 'tools': 0.75}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'name': 'seed-synth', 'acc': 0.51, 'macro_f1': 0.378, 'sentinel': 0.75, 'reasoning': 0.8, 'tools': 0.875}

name           acc    macroF1  sentinel  reasoning  tools
base           0.515  0.395    0.917     0.800      0.750
seed           0.515  0.367    0.833     0.900      0.750
seed-synth     0.510  0.378    0.750     0.800      0.875


In [6]:
import shutil
for n in ["seed", "seed-synth"]:
    shutil.make_archive(f"{OUT}/lora-{n}", "zip", f"{OUT}/lora-{n}")
print("Download lora-*.zip and result_*.json from the Output tab.")

Download lora-*.zip and result_*.json from the Output tab.
